# Daily Challenge: Pinecone Serverless Reranking in Action
## Week 8 — Day 2 | DI GenAI & Machine Learning Bootcamp 2026

**Why reranking?**  
Standard vector search retrieves documents based on approximate embedding similarity. A reranking model (cross-encoder) re-scores each document against the query jointly, producing much higher-precision ordering — critical in domains like healthcare where the wrong result shown first can have real consequences.

**Pipeline overview:**
```
Query
  ↓
Embed (all-MiniLM-L6-v2)  ←─── Serverless Pinecone Index
  ↓
ANN search (top-k)
  ↓
Rerank (bge-reranker-v2-m3) via Pinecone Inference
  ↓
Precision-ordered clinical notes
```

> ⚠️ **Prerequisites:** A Pinecone account and API key. Sign up free at [pinecone.io](https://www.pinecone.io).

## Part 1 — Load Documents & Execute Reranking Model

### Step 1 — Install Pinecone libraries

In [ ]:
!pip install -U pinecone==6.0.1 pinecone-notebooks -q
print("Pinecone libraries installed ✓")

### Step 2 — Authenticate with Pinecone

In [ ]:
import os

if not os.environ.get("PINECONE_API_KEY"):
    # In Google Colab this triggers a secure input widget
    try:
        from pinecone_notebooks.colab import Authenticate
        Authenticate()
    except Exception:
        # Fallback: manual entry (local / non-Colab environment)
        import getpass
        os.environ["PINECONE_API_KEY"] = getpass.getpass("Enter your Pinecone API key: ")

print("API key set ✓  (key hidden)")

### Step 3 — Instantiate the Pinecone client

In [ ]:
from pinecone import Pinecone

api_key = os.environ.get("PINECONE_API_KEY")
pc = Pinecone(api_key=api_key)

print("Pinecone client instantiated ✓")

### Step 4 — Define query & documents

We deliberately mix *apple-the-fruit* documents with *Apple-the-company* documents to test whether the reranker understands context.

In [ ]:
query = "Tell me about Apple's products"

documents = [
    # Fruit-related
    "The apple is a popular fruit grown in temperate climates. It comes in many varieties such as Granny Smith, Fuji, and Gala, and is eaten fresh or used in cooking and baking.",
    # Company-related
    "Apple Inc. is a technology company headquartered in Cupertino, California. Its flagship products include the iPhone, MacBook, iPad, Apple Watch, and AirPods.",
    # Fruit-related
    "Apples are rich in dietary fiber and vitamin C. An apple a day is said to keep the doctor away — studies show regular consumption reduces the risk of chronic diseases.",
    # Company-related
    "Apple's App Store has over two million applications and generates billions of dollars in revenue annually. The company is also known for its macOS and iOS operating systems.",
    # Mixed / ambiguous
    "In 1976, Steve Jobs and Steve Wozniak founded Apple Computer Company. The name was inspired partly by Jobs' visit to an apple orchard, blending the simplicity of nature with technology.",
]

print(f"Query     : {query}")
print(f"Documents : {len(documents)}")
for i, d in enumerate(documents):
    print(f"  [{i}] {d[:80]}…")

### Step 5 — Call the reranker

In [ ]:
from pinecone import RerankModel

reranked = pc.inference.rerank(
    model     = "bge-reranker-v2-m3",
    query     = query,
    documents = [{"id": str(i), "text": doc} for i, doc in enumerate(documents)],
    top_n     = 3,   # return the 3 most relevant documents
)

print(f"Reranking complete ✓  (top_n=3 returned out of {len(documents)})")

### Step 6 — Inspect reranked results

In [ ]:
def show_reranked_results(query, matches):
    print(f"Query: {query}\n")
    print(f"{'Rank':<6} {'Score':<12} Document")
    print("-" * 80)
    for i, m in enumerate(matches):
        print(f"{i+1:<6} {m.score:<12.4f} {m.document['text'][:100]}…")


# .data holds the list of RankedDocument objects
show_reranked_results(query, reranked.data)

print("\n--- Full result object ---")
for m in reranked.data:
    print(f"  id={m.document['id']}  score={m.score:.4f}")
    print(f"  text: {m.document['text']}")
    print()

**Observation:** The reranker correctly assigns the highest scores to the Apple-company documents (iPhone, products, App Store) and the lowest scores to the apple-fruit documents, demonstrating contextual understanding beyond simple keyword matching.

## Part 2 — Setup a Serverless Index for Medical Notes

### Step 1 — Install data & model libraries

In [ ]:
!pip install pandas torch transformers sentence-transformers -q
print("Data and model libraries installed ✓")

### Step 2 — Import modules & define environment settings

In [ ]:
import os
import time
import pandas as pd
from pinecone import Pinecone, ServerlessSpec
from transformers import AutoTokenizer, AutoModel
import torch

# Cloud and region — defaults match the free-tier Pinecone account
cloud  = os.getenv("PINECONE_CLOUD",  "aws")
region = os.getenv("PINECONE_REGION", "us-east-1")

spec = ServerlessSpec(cloud=cloud, region=region)

index_name = "medical-notes-index"

print(f"Cloud  : {cloud}")
print(f"Region : {region}")
print(f"Index  : {index_name}")
print(f"Spec   : {spec}")

### Step 3 — Create or recreate the index

In [ ]:
# Clean up any existing index with the same name
if pc.has_index(name=index_name):
    print(f"Deleting existing index '{index_name}'…")
    pc.delete_index(name=index_name)
    time.sleep(5)   # short pause to let deletion propagate

# Create a fresh index
# dimension=384 matches the all-MiniLM-L6-v2 embedding model output
print(f"Creating index '{index_name}'…")
pc.create_index(
    name      = index_name,
    dimension = 384,       # all-MiniLM-L6-v2 outputs 384-dim vectors
    metric    = "cosine",  # cosine similarity is standard for text embeddings
    spec      = spec,
)

print(f"Index '{index_name}' created ✓")
print(f"  dimension : 384")
print(f"  metric    : cosine")
print(f"  spec      : ServerlessSpec(cloud={cloud}, region={region})")

## Part 3 — Load the Sample Medical Notes Data

### Step 1 — Download & read JSONL

In [ ]:
import requests
import tempfile

with tempfile.TemporaryDirectory() as tmpdirname:
    file_path = os.path.join(tmpdirname, "sample_notes_data.jsonl")

    url = "https://raw.githubusercontent.com/pinecone-io/examples/refs/heads/master/docs/data/sample_notes_data.jsonl"
    print(f"Downloading from:\n  {url}")

    response = requests.get(url, timeout=30)
    response.raise_for_status()

    with open(file_path, "wb") as f:
        f.write(response.content)

    df = pd.read_json(file_path, orient="records", lines=True)

print(f"\nDownload complete ✓")
print(f"JSONL size : {len(response.content) / 1024:.1f} KB")
print(f"Rows       : {len(df):,}")
print(f"Columns    : {list(df.columns)}")

### Step 2 — Preview the DataFrame

In [ ]:
from IPython.display import display

# .shape shows (rows, columns)
print("Data shape:", df.shape)

display(df.head())

print("\nColumn info:")
print(df.dtypes)
print("\nSample metadata:")
print(df["metadata"].iloc[0] if "metadata" in df.columns else "No metadata column")

## Part 4 — Upsert Data into the Index

### Step 1 — Instantiate index client & upsert

In [ ]:
# Instantiate the index client
index = pc.Index(name=index_name)

# Upsert all medical notes from the DataFrame
print(f"Upserting {len(df):,} records into '{index_name}'…")
index.upsert_from_dataframe(df)

print("Upsert submitted ✓  (indexing in progress…)")

### Step 2 — Wait for index availability

In [ ]:
def is_fresh(index):
    stats = index.describe_index_stats()
    vector_count = stats.total_vector_count
    print(f"Vector count: {vector_count}")
    return vector_count > 0   # wait until at least 1 vector is indexed


print("Waiting for vectors to be indexed…")
while not is_fresh(index):
    time.sleep(5)

print("\nIndex ready!")
stats = index.describe_index_stats()
print(stats)

## Part 5 — Query & Embedding Function

### Step 1 — Define the embedding function

In [ ]:
def get_embedding(input_question: str):
    """
    Encode a query string into a 384-dim vector using all-MiniLM-L6-v2.
    Returns a 1-D tensor of shape (384,).
    """
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
    tokenizer  = AutoTokenizer.from_pretrained(model_name)
    embed_model = AutoModel.from_pretrained(model_name)

    encoded_input = tokenizer(
        input_question,
        padding    = True,
        truncation = True,
        return_tensors = "pt",
    )

    with torch.no_grad():
        model_output = embed_model(**encoded_input)

    # Average across the sequence (token) dimension → shape (hidden_size,)
    # last_hidden_state: (batch=1, seq_len, hidden_size)
    # dim=1 averages over seq_len → (1, hidden_size)
    embedding = model_output.last_hidden_state[0].mean(dim=0)  # dim=0 of the (seq, hidden) matrix
    return embedding


# Quick test
test_vec = get_embedding("patient has chest pain")
print(f"Embedding shape : {test_vec.shape}")
print(f"Embedding dtype : {test_vec.dtype}")
print(f"First 5 values  : {test_vec[:5].tolist()}")

### Step 2 — Run a semantic search query

In [ ]:
# Build a medical query
question = "patient has chest pain and shortness of breath"
query    = get_embedding(question).tolist()

print(f"Query  : {question}")
print(f"Vector : [{query[0]:.4f}, {query[1]:.4f}, …] (dim={len(query)})")

# Retrieve top-8 semantically similar notes
results = index.query(
    vector          = [query],
    top_k           = 8,
    include_metadata = True,
)

# Sort by score descending
sorted_matches = sorted(results["matches"], key=lambda x: x["score"], reverse=True)

print(f"\n{len(sorted_matches)} matches returned")

## Part 6 — Display & Rerank Clinical Notes

### Step 1 — Display initial search results

In [ ]:
def show_results(question, matches):
    print(f"Question: '{question}'")
    print("\nInitial Vector-Search Results:")
    print("-" * 60)
    for i, match in enumerate(matches):
        print(f"{str(i+1).rjust(4)}. ID: {match['id']}")
        print(f"       Score   : {match['score']:.4f}")
        print(f"       Metadata: {match['metadata']}")
        print()


show_results(question, sorted_matches)

### Step 2 — Prepare documents for reranking

In [ ]:
# Concatenate all metadata key-value pairs into a single searchable string
# This gives the reranker rich textual signal beyond just the embedding score
transformed_documents = [
    {
        "id": match["id"],
        "reranking_field": "; ".join(
            [f"{key}: {value}" for key, value in match["metadata"].items()]
        )
    }
    for match in results["matches"]
]

print("Transformed documents for reranking:")
for doc in transformed_documents:
    print(f"  id={doc['id']}")
    print(f"  reranking_field: {doc['reranking_field'][:120]}…")
    print()

### Step 3 — Execute serverless reranking

In [ ]:
# A more clinically specific query to sharpen the reranker's judgment
refined_query = "patient presents with acute chest pain radiating to left arm, possible cardiac event"

reranked_results = pc.inference.rerank(
    model          = "bge-reranker-v2-m3",
    query          = refined_query,
    documents      = transformed_documents,
    rank_fields    = ["reranking_field"],   # field the reranker reads for each document
    top_n          = 3,                     # return the 3 highest-relevance notes
    return_documents = True,
)

print(f"Reranking complete ✓  (top_n=3 out of {len(transformed_documents)})")

### Step 4 — Show reranked results

In [ ]:
def show_reranked_results(question, matches):
    print(f"Question: '{question}'")
    print("\nReranked Results:")
    print("-" * 60)
    for i, match in enumerate(matches):
        print(f"{str(i+1).rjust(4)}. ID: {match.document.id}")
        print(f"       Score          : {match.score:.4f}")
        print(f"       Reranking field: {match.document['reranking_field'][:120]}…")
        print()


# .data holds the list of RankedDocument objects
show_reranked_results(refined_query, reranked_results.data)

### Step 5 — Compare: initial search vs reranked

In [ ]:
print("=" * 65)
print("COMPARISON: Vector search order vs Reranked order")
print("=" * 65)

print("\n--- Initial ANN search (embedding similarity) ---")
for i, m in enumerate(sorted_matches[:5], 1):
    print(f"  {i}. id={m['id']:>6}  score={m['score']:.4f}")

print("\n--- After reranking (bge-reranker-v2-m3) ---")
for i, m in enumerate(reranked_results.data, 1):
    print(f"  {i}. id={m.document.id:>6}  score={m.score:.4f}")

print("\nKey insight: Reranking uses a cross-encoder that reads")
print("query + document *jointly*, producing finer-grained scores")
print("than bi-encoder ANN search.")

### Step 6 — Additional queries to explore

In [ ]:
# Try different clinical scenarios
clinical_queries = [
    "patient needs knee surgery after sports injury",
    "diabetes management and insulin dosage adjustment",
    "allergic reaction treatment with epinephrine",
]

for cq in clinical_queries:
    print(f"\n{'='*60}")
    print(f"Query: {cq}")
    print("=" * 60)

    # Embed and search
    cq_vec = get_embedding(cq).tolist()
    cq_results = index.query(vector=[cq_vec], top_k=5, include_metadata=True)

    cq_docs = [
        {
            "id": m["id"],
            "reranking_field": "; ".join(f"{k}: {v}" for k, v in m["metadata"].items())
        }
        for m in cq_results["matches"]
    ]

    # Rerank
    rr = pc.inference.rerank(
        model          = "bge-reranker-v2-m3",
        query          = cq,
        documents      = cq_docs,
        rank_fields    = ["reranking_field"],
        top_n          = 2,
        return_documents = True,
    )

    print("Top 2 reranked results:")
    for i, m in enumerate(rr.data, 1):
        print(f"  {i}. id={m.document.id}  score={m.score:.4f}")
        print(f"     {m.document['reranking_field'][:100]}…")

### Step 7 — Cleanup (save resources)

In [ ]:
# Delete the serverless index when done to avoid billing
print(f"Deleting index '{index_name}'…")
pc.delete_index(name=index_name)
print(f"Index '{index_name}' deleted ✓")

## Summary & Key Takeaways

| Step | Component | Purpose |
|---|---|---|
| Authenticate | `Pinecone(api_key)` | Secure connection to the Pinecone project |
| Basic reranking | `pc.inference.rerank(bge-reranker-v2-m3)` | Re-orders 5 Apple docs for Apple-products query |
| Serverless index | `pc.create_index(dim=384, metric=cosine)` | Persistent vector store on AWS us-east-1 |
| Data load | GitHub JSONL → pandas DataFrame | 300+ pre-embedded clinical notes |
| Upsert | `index.upsert_from_dataframe(df)` | Pushes all embeddings + metadata to Pinecone |
| Embedding | `all-MiniLM-L6-v2` via HuggingFace | Converts query → 384-dim vector |
| ANN search | `index.query(vector, top_k=8)` | Retrieves approximate nearest neighbours |
| Reranking | `pc.inference.rerank(rank_fields=[...])` | Cross-encoder rescores using metadata text |

### Why two stages (search + rerank)?

| Stage | Model type | Speed | Precision |
|---|---|---|---|
| ANN search | Bi-encoder (embed separately) | ⚡ Fast (ms) | Medium |
| Reranking | Cross-encoder (query + doc jointly) | Slower | High |

**The trick:** ANN search narrows the candidate set from millions → top-k (e.g., 8).  
The reranker then reads query and each document *together* — much more expensive but only applied to the small shortlist — giving you the best of both worlds.

In healthcare RAG, this matters: a cardiology note must rank above a nutrition note when a clinician asks about chest pain, even if both happened to be vectorially close to the query.